<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/logo_curso.png" width="300">

# Python para Actuarios

## Día 1, bloque 2: Excel y Python

| Día | Horario | Duración |
|---|---|---|
| Miércoles 11 de noviembre | 11:30 a 13:00 | 90 minutos |

**Objetivos**

1. Traducir las operaciones habituales de Excel a su equivalente en pandas.
2. Leer libros de Excel con varias hojas, filas de título y rangos concretos.
3. Leer ficheros CSV exportados con configuración española.
4. Construir tablas dinámicas y triángulos con `pivot_table`.
5. Cruzar tablas con controles de cuadre.
6. Escribir los resultados en un libro de Excel con varias hojas.

**Cómo trabajar con este notebook**

1. El código que debes completar está marcado con `***`. Sustitúyelo y ejecuta la celda con `Mayús + Intro`.
2. Ejecuta las celdas en orden: muchas utilizan variables definidas en celdas anteriores.
3. Si Colab reinicia el entorno, ejecuta de nuevo la preparación del entorno y las celdas anteriores (*Entorno de ejecución > Ejecutar anteriores*).

**Datos del bloque**

Una cartera sintética de Hogar con cierre a 31/12/2024, en dos ficheros:

1. `cierre_hogar.xlsx`, con tres hojas: `Cartera` (una fila por póliza), `Siniestros` (con filas de título antes de la cabecera) y `Parametros` (una tabla pequeña en un rango concreto).
2. `primas_hogar.csv`, la nueva producción de 2024 (prima anual de las pólizas con efecto en cada mes), exportada desde un Excel configurado en español.

**Contenido**

1. Equivalencias entre Excel y pandas
2. Leer libros de Excel
3. Leer CSV en formato español
4. Tablas dinámicas y triángulos
5. Cruces con control de cuadre
6. Escribir resultados en Excel

## Preparación del entorno

Ejecuta esta celda antes que cualquier otra. En Google Colab descarga los datos del curso y sitúa el notebook en la carpeta correcta; en una instalación local no hace nada. Si Colab reinicia el entorno, vuelve a ejecutarla.

In [ ]:
# Preparación del entorno: en Google Colab descarga el repositorio del curso (en local no hace nada)
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/python_actuarios"):
        !git clone -q --depth 1 https://github.com/PrimeReSolutions/python_actuarios.git /content/python_actuarios
    %cd /content/python_actuarios/notebooks

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")
SALIDAS = Path("../salidas") if Path("../datos").exists() else Path("salidas")
SALIDAS.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Equivalencias entre Excel y pandas

Casi todo lo que se hace a diario en Excel tiene una instrucción directa en pandas. La diferencia es que en pandas cada paso queda escrito, se puede repetir en el siguiente cierre sin rehacerlo a mano y funciona igual con 100 filas que con 10 millones.

| En Excel | En pandas |
|---|---|
| Abrir un libro o importar un CSV | `pd.read_excel`, `pd.read_csv` |
| Autofiltro | `df.loc[condición]` |
| Columna con una fórmula arrastrada | `df["nueva"] = df["a"] * df["b"]` |
| `SI` (`IF`) | `np.where(condición, valor_si, valor_no)` |
| `SUMAR.SI.CONJUNTO` (`SUMIFS`), `CONTAR.SI.CONJUNTO` (`COUNTIFS`) | `df.groupby(...).agg(...)` |
| Tabla dinámica | `df.pivot_table(...)` |
| `BUSCARX` (`XLOOKUP`), `BUSCARV` (`VLOOKUP`) | `df.merge(...)` |
| Quitar duplicados | `df.drop_duplicates()` |
| Ordenar | `df.sort_values(...)` |
| Copiar una tabla debajo de otra | `pd.concat([...])` |
| Guardar como libro de Excel | `df.to_excel(...)` |

## 2. Leer libros de Excel

### 2.1 Ver las hojas de un libro

`pd.ExcelFile` abre el libro sin leer los datos y permite consultar el nombre de sus hojas.

In [ ]:
libro = pd.ExcelFile(DATOS / "cierre_hogar.xlsx")
libro.sheet_names

### 2.2 Leer una hoja

`pd.read_excel` lee una hoja con el argumento `sheet_name`. Si la primera fila contiene los nombres de columna, no hace falta nada más. Las fechas guardadas como fecha en Excel llegan ya como fecha (`datetime64`).

### Ejercicio 2.1 Leer la cartera

Lee la hoja `Cartera` y comprueba su tamaño y el tipo de cada columna.

In [ ]:
cartera = pd.read_excel(DATOS / "cierre_hogar.xlsx", sheet_name="***")

print(cartera.***)
cartera.dtypes

In [ ]:
cartera.head()

### 2.3 Cabeceras que no están en la primera fila

Es habitual recibir hojas con un título, una fecha o una nota antes de la tabla. Abre la hoja `Siniestros`: la cabecera está en la fila 4 de Excel. El argumento `header` indica en qué fila está la cabecera, contando desde 0, de modo que la fila 4 de Excel es `header=3`.

### Ejercicio 2.2 Leer los siniestros

Lee primero la hoja sin indicar la cabecera y observa el resultado. Después léela correctamente.

In [ ]:
# Lectura sin indicar la cabecera: las filas de título se mezclan con los datos
pd.read_excel(DATOS / "cierre_hogar.xlsx", sheet_name="Siniestros").head()

In [ ]:
siniestros = pd.read_excel(DATOS / "cierre_hogar.xlsx", sheet_name="Siniestros", header=***)

print(siniestros.shape)
siniestros.head()

### 2.4 Leer un rango concreto

La hoja `Parametros` tiene una tabla pequeña en el rango `B3:C7`. Para leer solo ese rango se combinan tres argumentos:

1. `usecols="B:C"`: columnas que se leen.
2. `skiprows=2`: filas que se saltan antes de la cabecera.
3. `nrows=4`: número de filas de datos.

### Ejercicio 2.3 Leer los parámetros del cierre

Lee el rango y conviértelo en un diccionario `{parametro: valor}`, de modo que cada parámetro pueda usarse por su nombre en el resto del notebook.

In [ ]:
param_df = pd.read_excel(
    DATOS / "cierre_hogar.xlsx",
    sheet_name="Parametros",
    usecols="***",
    skiprows=***,
    nrows=***,
)
param_df

In [ ]:
parametros = dict(zip(param_df["***"], param_df["***"]))
parametros

### Ejercicio 2.4 Clasificar siniestros graves

`np.where` es el equivalente de la función `SI` de Excel: evalúa una condición fila a fila y devuelve un valor u otro. Marca como `"Grave"` los siniestros cuyo coste total (pagado más provisión) supera el umbral de la hoja de parámetros, y como `"Atricional"` el resto.

In [ ]:
siniestros["coste_total"] = siniestros["***"] + siniestros["***"]

umbral = parametros["***"]
siniestros["tipo"] = np.where(siniestros["coste_total"] > ***, "Grave", "Atricional")

siniestros["tipo"].value_counts()

### 2.5 Leer todas las hojas a la vez

Con `sheet_name=None`, `read_excel` lee todas las hojas y devuelve un diccionario cuyas claves son los nombres de las hojas. Es útil cuando un libro tiene una hoja por mes, por ramo o por entidad.

In [ ]:
todas = pd.read_excel(DATOS / "cierre_hogar.xlsx", sheet_name=None)

for nombre, hoja in todas.items():
    print(f"{nombre}: {hoja.shape[0]} filas y {hoja.shape[1]} columnas")

## 3. Leer CSV en formato español

Un Excel configurado en español exporta los CSV con punto y coma como separador de columnas, coma decimal, punto como separador de miles y fechas en formato `dd/mm/aaaa`. `pd.read_csv` espera por defecto el formato anglosajón, así que hay que indicárselo. Esta es una de las causas más frecuentes de importes leídos como texto o multiplicados por mil.

In [ ]:
# Lectura con las opciones por defecto: todo queda en una sola columna
pd.read_csv(DATOS / "primas_hogar.csv").head(3)

### Ejercicio 3.1 Leer el CSV correctamente

Indica el separador de columnas, el separador decimal y el separador de miles. Después convierte la columna `mes` en fecha con el formato día, mes y año.

In [ ]:
primas = pd.read_csv(
    DATOS / "primas_hogar.csv",
    sep="***",
    decimal="***",
    thousands="***",
)
primas["mes"] = pd.to_datetime(primas["mes"], format="***")

print(primas.dtypes)
primas.head()

### Ejercicio 3.2 Cuadrar el CSV con la cartera

El CSV debería contener la prima anual de las pólizas con efecto en 2024. Comprueba que el total coincide con el que se obtiene de la cartera. `.dt.year` extrae el año de una columna de fechas.

In [ ]:
total_csv = primas["***"].sum()
total_cartera = cartera.loc[cartera["fecha_efecto"].dt.year == ***, "prima_anual"].sum()

print(f"Total CSV:     {total_csv:,.2f}")
print(f"Total cartera: {total_cartera:,.2f}")
print("Cuadra" if abs(total_csv - total_cartera) < 0.01 else "No cuadra")

## 4. Tablas dinámicas y triángulos

`pivot_table` reproduce una tabla dinámica de Excel:

| Tabla dinámica de Excel | Argumento de `pivot_table` |
|---|---|
| Filas | `index` |
| Columnas | `columns` |
| Valores | `values` |
| Resumir valores por | `aggfunc` (`"sum"`, `"count"`, `"mean"`...) |
| Totales generales | `margins=True` |

### Ejercicio 4.1 Prima por provincia y producto

Construye una tabla con una fila por provincia, una columna por producto, la suma de la prima anual y los totales generales.

In [ ]:
prima_prov = cartera.pivot_table(
    index="***",
    columns="***",
    values="***",
    aggfunc="***",
    margins=True,
    margins_name="Total",
)
prima_prov

### 4.2 Del formato largo al triángulo

Los sistemas suelen entregar la información de siniestros en formato largo: una fila por año de ocurrencia y periodo de desarrollo. Un triángulo de desarrollo es exactamente una tabla dinámica de esos datos, con el año de ocurrencia en las filas y el retraso (`lag`) en las columnas.

### Ejercicio 4.2 Triángulo de pagos acumulados

Con `auto.csv`, construye el triángulo de pagos acumulados de *PersonalAuto*.

In [ ]:
auto = pd.read_csv(DATOS / "auto.csv", thousands=",")
auto["lag"] = auto["development"] - auto["origin"]
personal = auto[auto["lob"] == "PersonalAuto"]

triangulo = personal.pivot_table(index="***", columns="***", values="***", aggfunc="sum")
triangulo

La división de cada columna entre la anterior da los factores de desarrollo individuales, punto de partida del método Chain Ladder que se trabajará en reservas no vida. `shift(axis=1)` desplaza el triángulo una columna a la derecha, de modo que cada celda queda alineada con la de su periodo anterior.

In [ ]:
factores = (triangulo / triangulo.shift(axis=1)).iloc[:, 1:]    # la columna 1 es el factor de 0 a 1
factores.round(3)

### Ejercicio 4.3 Triángulo de número de siniestros de Hogar

Construye el triángulo de número de siniestros de la cartera de Hogar, con el año de ocurrencia en las filas y el retraso de declaración en años en las columnas. Es la información de partida para estimar los siniestros ocurridos y no declarados.

In [ ]:
siniestros["anio_ocurrencia"] = siniestros["fecha_ocurrencia"].dt.***
siniestros["retraso"] = siniestros["fecha_declaracion"].dt.year - siniestros["***"]

triangulo_n = siniestros.pivot_table(
    index="***",
    columns="***",
    values="siniestro",
    aggfunc="***",
)
triangulo_n

## 5. Cruces con control de cuadre

En Excel, un `BUSCARX` que no encuentra la clave devuelve un error visible en la celda. En pandas, un `merge` mal planteado no da error: puede perder filas o, peor, duplicarlas y multiplicar los importes sin aviso. Por eso todo cruce debe ir acompañado de controles. pandas ofrece dos argumentos para ello:

1. `indicator=True` añade la columna `_merge`, que indica si cada fila encontró correspondencia (`both`) o solo está en una de las tablas (`left_only`, `right_only`).
2. `validate` comprueba la relación esperada entre las claves y detiene el cruce si no se cumple: `"one_to_one"`, `"many_to_one"`, `"one_to_many"`.

### 5.1 Qué ocurre con una clave duplicada

En este ejemplo la póliza `P2` aparece dos veces en la tabla de pólizas, por un error de carga. El cruce duplica su siniestro y el importe total pasa de 1 500 a 2 000 sin ningún aviso.

In [ ]:
sin_ej = pd.DataFrame({"poliza": ["P1", "P2"], "importe": [1000, 500]})
pol_ej = pd.DataFrame({"poliza": ["P1", "P2", "P2"], "producto": ["Hogar Básico", "Hogar Plus", "Hogar Plus"]})

cruce_ej = sin_ej.merge(pol_ej, on="poliza", how="left")
print("Importe antes del cruce: ", sin_ej["importe"].sum())
print("Importe después del cruce:", cruce_ej["importe"].sum())

In [ ]:
# Con validate, el mismo cruce se detiene y explica el motivo
try:
    sin_ej.merge(pol_ej, on="poliza", how="left", validate="many_to_one")
except Exception as error:
    print(type(error).__name__, ":", error)

### Ejercicio 5.1 Cruzar siniestros con la cartera

Cada póliza puede tener varios siniestros, pero cada siniestro corresponde a una sola póliza. Cruza los siniestros con la cartera conservando todos los siniestros, valida la relación entre claves y añade el indicador de correspondencia.

In [ ]:
sin_cartera = siniestros.merge(
    cartera[["poliza", "producto", "provincia", "canal"]],
    on="poliza",
    how="***",
    validate="***",
    indicator=***,
)
sin_cartera["_merge"].value_counts()

### Ejercicio 5.2 Controles del cruce

1. Comprueba que el número de siniestros y el importe pagado son los mismos antes y después del cruce.
2. Lista los siniestros cuya póliza no existe en la cartera. En un cierre real, son incidencias que hay que comunicar al área responsable de los datos.

In [ ]:
print("Siniestros antes y después:", len(siniestros), len(***))
print("Pagado antes:   ", round(siniestros["pagado"].sum(), 2))
print("Pagado después: ", round(***["pagado"].sum(), 2))

sin_poliza = sin_cartera[sin_cartera["_merge"] == "***"]
sin_poliza[["siniestro", "poliza", "fecha_ocurrencia", "pagado", "provision"]]

### Ejercicio 5.3 Siniestralidad por producto

Calcula, para cada producto, la siniestralidad del año de ocurrencia 2024: el coste de los siniestros ocurridos en 2024 entre la prima anual de la cartera, que está toda en vigor durante ese año. Primero se agrega cada tabla por producto y después se cruzan los dos resúmenes, que tienen un único registro por producto.

La siniestralidad obtenida es la de los siniestros ya declarados; los ocurridos y aún no declarados la aumentarán, como muestra el triángulo del ejercicio 4.3.

In [ ]:
prima_producto = cartera.groupby("producto").agg(prima=("prima_anual", "***"))

sin_2024 = sin_cartera[sin_cartera["anio_ocurrencia"] == ***]
coste_producto = sin_2024.groupby("producto").agg(
    n_siniestros=("siniestro", "***"),
    coste=("coste_total", "***"),
)

resumen = prima_producto.merge(coste_producto, on="producto", how="left", validate="***")
resumen["siniestralidad"] = resumen["***"] / resumen["***"]
resumen

## 6. Escribir resultados en Excel

`df.to_excel` guarda una tabla en un libro. Para escribir varias tablas en un mismo libro, cada una en su hoja, se usa `pd.ExcelWriter` dentro de un bloque `with`, que cierra y guarda el fichero al terminar.

### Ejercicio 6.1 Informe del cierre

Genera un libro con cuatro hojas: el resumen por producto, la prima por provincia, el triángulo de número de siniestros y los siniestros sin póliza.

In [ ]:
ruta_informe = SALIDAS / "informe_cierre_hogar.xlsx"

with pd.ExcelWriter(ruta_informe) as writer:
    resumen.to_excel(writer, sheet_name="Resumen producto")
    ***.to_excel(writer, sheet_name="Prima provincia")
    ***.to_excel(writer, sheet_name="Triangulo numero")
    ***.to_excel(writer, sheet_name="Sin poliza", index=False)

pd.ExcelFile(ruta_informe).sheet_names

En Google Colab el fichero se guarda en la máquina temporal del entorno. La celda siguiente lo descarga a tu equipo; en una instalación local, el fichero ya está en la carpeta `salidas`.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(ruta_informe))

## Resumen

1. Cada operación habitual de Excel tiene un equivalente directo en pandas, que queda escrito y se puede repetir en cada cierre.
2. `read_excel` lee hojas concretas, salta filas de título (`header`, `skiprows`) y lee rangos (`usecols`, `nrows`); con `sheet_name=None` lee el libro completo.
3. Los CSV exportados desde un Excel en español requieren `sep=";"`, `decimal=","` y `thousands="."`.
4. `pivot_table` construye tablas dinámicas y triángulos de desarrollo.
5. Todo cruce debe comprobarse: `validate` e `indicator=True`, y los totales antes y después.
6. `pd.ExcelWriter` genera un informe con varias hojas en un solo paso.

En la segunda parte del bloque trabajaremos directamente en Excel con Python en Excel.